# 📧 Spam Detection Project


In [ ]:
import pandas as pd
import numpy as np
import string
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import nltk

nltk.download('stopwords')

## Load Dataset

In [ ]:
df = pd.read_csv('spam_ham_dataset.csv')
df.head()

In [ ]:
print(df.shape)
print(df.isnull().sum())
print(df.duplicated().sum())

## Visualization

In [ ]:
#Class Distribution
import seaborn as sns

sns.countplot(x=df['label'])
plt.title("Spam vs Ham Distribution")
plt.show()

In [ ]:
#Message Length Analysis
df['length'] = df['text'].apply(len)

plt.hist(df[df['label']==0]['length'], bins=50, alpha=0.7, label='Ham')
plt.hist(df[df['label']==1]['length'], bins=50, alpha=0.7, label='Spam')

plt.legend()
plt.title("Message Length Distribution")
plt.show()

In [ ]:
# Visualize text length distribution
df['text_length'] = df['text'].apply(len)
plt.figure(figsize=(8, 5))
sns.histplot(df['text_length'], bins=50)
plt.title('Distribution of Email Text Lengths')
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.show()

## Data Cleaning

In [ ]:
#
punctuations_list = string.punctuation
def remove_punctuations(text):
    temp = str.maketrans('', '', punctuations_list)
    return text.translate(temp)

In [ ]:
# Remove stopwords from the text
def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))

    imp_words = []
    # Storing the important words
    for word in str(text).split():
        word = word.lower()
        if word not in stop_words:
            imp_words.append(word)
    output = " ".join(imp_words)
    return output

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    if(text.startswith('subject')):
        text = text[7:]
    text = remove_punctuations(text)
    text = remove_stopwords(text)
    return text
df['text'] = df['text'].apply(clean_text)
df.head()

In [ ]:
#WordCloud
from wordcloud import WordCloud
def plot_word_cloud(data, typ):
    email_corpus = " ".join(data['text'])
    wc = WordCloud(background_color='black', max_words=100, width=800, height=400).generate(email_corpus)
    plt.figure(figsize=(7, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f'WordCloud for {typ} Emails', fontsize=15)
    plt.axis('off')
    plt.show()

plot_word_cloud(df[df['label'] == 'ham'], typ='Non-Spam')
plot_word_cloud(df[df['label'] == 'spam'], typ='Spam')

## Encode Labels

In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

## Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['text'])
y = df['label']

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Build Model

In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

## Train Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

In [ ]:
#Training History
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.legend()
plt.title("Model Accuracy")
plt.show()
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Model Loss")
plt.show()

## Evaluate Model

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Test Custom Email

In [ ]:
def predict_email(text):
    text = clean_text(text)
    vector = vectorizer.transform([text])
    pred = model.predict(vector)[0][0]
    return "Spam" if pred > 0.5 else "Ham"

predict_email("Congratulations! You won a free prize!")

## Save Model

In [ ]:
import joblib

model.save("model.h5")
joblib.dump(vectorizer, "vectorizer.pkl")